# stat.ipynb

## Úvod

Tento notebook obsahuje dvě hypotézy, které testujeme na základě datasetů `accidents.pkl.gz` a `vehicles.pkl.gz`.

- **Hypotéza 1**:  
  "Nehody se následky na zdraví jsou na silnicích I. třídy stejně pravděpodobné jako na dálnicích."

- **Hypotéza 2**:  
  "Škoda při nehodách trolejbusů je nižší než při nehodách autobusů a tato odchylka je statisticky významná."

----

In [ ]:
# Buňka [1]: Import knihoven a načtení dat

import pandas as pd
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

# Načtení datasetů (soubor accidents.pkl.gz, vehicles.pkl.gz musí být v aktuálním adresáři)
df_accidents = pd.read_pickle("accidents.pkl.gz")
df_vehicles = pd.read_pickle("vehicles.pkl.gz")

print("df_accidents.shape:", df_accidents.shape)
print("df_vehicles.shape:", df_vehicles.shape)

print("Sloupce df_accidents:", df_accidents.columns)
print("Sloupce df_vehicles:", df_vehicles.columns)

## Hypotéza 1

**Na silnicích I. třídy (p36=1) a dálnicích (p36=0) jsou nehody s následky (p9=1) stejně pravděpodobné.**

Datový slovník říká:
- `p9=1`: nehoda s následky na životě
- `p9=2`: nehoda pouze s hmotnou škodou
- `p36=0`: dálnice
- `p36=1`: silnice I. třídy

Pro otestování použijeme χ² test nezávislosti (kontingenční tabulka 2×2).


In [20]:
# --- Buňka 2: Příprava dat pro hypotézu 1 ---

# 1) Vyhodíme záznamy, kde p9 nebo p36 chybí (NaN):
df_accidents_clean = df_accidents.dropna(subset=["p9", "p36"])

# 2) Definice filtru na dálnice (p36=0) a silnice I. třídy (p36=1)
mask_dalnice  = (df_accidents_clean["p36"] == 0)
mask_silnice1 = (df_accidents_clean["p36"] == 1)

# 3) "nehoda s následky" definujeme, pokud p9 == 1
#    (tj. nehoda s následky na životě). Jinak p9 == 2 => nehoda bez následků.
df_accidents_clean["has_injury"] = (df_accidents_clean["p9"] == 1)

# 4) Vyčíslíme počty pro čtyři kombinace:
n_dalnice_injury     = sum(mask_dalnice  & (df_accidents_clean["has_injury"] == True))
n_dalnice_no_injury  = sum(mask_dalnice  & (df_accidents_clean["has_injury"] == False))

n_silnice1_injury    = sum(mask_silnice1 & (df_accidents_clean["has_injury"] == True))
n_silnice1_no_injury = sum(mask_silnice1 & (df_accidents_clean["has_injury"] == False))

print("Počet nehod na dálnicích s následky:",    n_dalnice_injury)
print("Počet nehod na dálnicích bez následků:",  n_dalnice_no_injury)
print("Počet nehod na silnicích I. třídy s následky:",   n_silnice1_injury)
print("Počet nehod na silnicích I. třídy bez následků:", n_silnice1_no_injury)

# 5) Kontingenční tabulka 2×2
dalnice_injury   = [n_dalnice_injury,   n_dalnice_no_injury]
silnice1_injury  = [n_silnice1_injury, n_silnice1_no_injury]

table = np.array([dalnice_injury, silnice1_injury])
print("\nKontingenční tabulka:\n", table)

# 6) Ověříme, zda ve sloupcích/řádcích není součet 0
col_sums = table.sum(axis=0)  # součty sloupců
row_sums = table.sum(axis=1)  # součty řádků

if 0 in col_sums or 0 in row_sums:
    print("\n[VAROVÁNÍ] Jedna z kategorií má nulový počet záznamů. Nelze provést chi2 test.")
    print("           (Zřejmě chybí nehody bez následků pro některý typ silnice.)")
else:
    # 7) Pokud máme nenulové počty, pokračujeme s chi2 testem
    chi2, pval, dof, expected = st.chi2_contingency(table)
    print(f"\nChi2-stat: {chi2:.4f}, p-value: {pval:.6f}, dof: {dof}")
    print("Očekávané četnosti:\n", expected)

    alpha = 0.05
    if pval < alpha:
        print("Zamítáme nulovou hypotézu: Pravděpodobnost nehody s následky se LIŠÍ.")
    else:
        print("Nezamítáme nulovou hypotézu: Pravděpodobnost nehody s následky je STEJNÁ.")


Počet nehod na dálnicích s následky: 1247
Počet nehod na dálnicích bez následků: 6674
Počet nehod na silnicích I. třídy s následky: 7059
Počet nehod na silnicích I. třídy bez následků: 14773

Kontingenční tabulka:
 [[ 1247  6674]
 [ 7059 14773]]

Chi2-stat: 794.1533, p-value: 0.000000, dof: 1
Očekávané četnosti:
 [[ 2211.26696468  5709.73303532]
 [ 6094.73303532 15737.26696468]]
Zamítáme nulovou hypotézu: Pravděpodobnost nehody s následky se LIŠÍ.


**Závěr k Hypotéze 1**  
- Pokud p-value < 0.05, na 5% hladině významnosti se liší pravděpodobnost nehod s následky.  
- Pokud p-value >= 0.05, test neprokázal rozdíl.  
- Pokud vyskočí varování „Jedna z kategorií má nulový počet záznamů“, v datasetu chybí kombinace (např. 0 nehod bez následků na dálnici), takže \(\chi^2\) test není smysluplný.

----

## Hypotéza 2

**„Škoda u trolejbusů (p44=11) je nižší než u autobusů (p44=8) a tato odchylka je statisticky významná“**  
Rozhodl jsme se pro Mann–Whitney test (jednostranný), protože data pravděpodobně nebudou normálně rozložená.


In [22]:
# Buňka [3]: Hypotéza 2 - zpracování vehicles.pkl.gz

df_vehicles_clean = df_vehicles.dropna(subset=["p44", "p53"])

# p44: 8=autobus, 11=trolejbus
mask_bus    = (df_vehicles_clean["p44"] == 8)
mask_trolej = (df_vehicles_clean["p44"] == 11)

damage_bus    = df_vehicles_clean.loc[mask_bus,    "p53"]
damage_trolej = df_vehicles_clean.loc[mask_trolej, "p53"]

print("Počet záznamů bus:", len(damage_bus))
print("Počet záznamů trolej:", len(damage_trolej))

print("\nPopis autobusů (p53):")
print(damage_bus.describe())
print("\nPopis trolejbusů (p53):")
print(damage_trolej.describe())

# Test normality
sample_bus   = damage_bus.sample(min(len(damage_bus), 2000))
sample_troj  = damage_trolej.sample(min(len(damage_trolej), 2000))
stat_b, p_b  = st.shapiro(sample_bus)
stat_t, p_t  = st.shapiro(sample_troj)
print(f"\nShapiro-Wilk bus: p={p_b:.5f}, trolej: p={p_t:.5f}")

# Mann-Whitney (jednostranný: trolej < bus)
stat, p_value = st.mannwhitneyu(damage_trolej, damage_bus, alternative="less")
print(f"\nMann-Whitney U-stat: {stat}, p-value={p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print("Zamítáme H0: Škoda u trolejbusů je průkazně NIŽŠÍ než u autobusů.")
else:
    print("Nelze zamítnout H0: test neprokázal nižší škodu trolejbusů.")


Počet záznamů bus: 4284
Počet záznamů trolej: 293

Popis autobusů (p53):
count     4284.000000
mean       440.688609
std       1051.772935
min          0.000000
25%         20.000000
50%        200.000000
75%        500.000000
max      30000.000000
Name: p53, dtype: float64

Popis trolejbusů (p53):
count      293.000000
mean       280.341297
std        803.918413
min          0.000000
25%          0.000000
50%         50.000000
75%        300.000000
max      10000.000000
Name: p53, dtype: float64

Shapiro-Wilk bus: p=0.00000, trolej: p=0.00000

Mann-Whitney U-stat: 469902.5, p-value=0.000000
Zamítáme H0: Škoda u trolejbusů je průkazně NIŽŠÍ než u autobusů.


**Závěr k Hypotéze 2**  
- Pokud p-value < 0.05, pak na hladině 5% **zamítáme H0** a říkáme: "Trolejbusy mají nižší škodu než autobusy, statisticky významně."  
- Pokud p-value >= 0.05, není rozdíl prokázán.  

----
